# NanniSCII · Step 6: pick and translate

Draws complaints at random from the cleaned archive sample, takes **one line from each**, translates it into Akkadian with `praeclarum/cuneiform` (through CuneiScribe), and converts it into Unicode cuneiform. The XXXX redactions are kept as breaks in the clay.

**Output:** `translations.csv`, the text the renderer draws its signs from, with a full log for every line (Akkadian, cuneiform, back-translation into English, gaps, sign coverage, validator verdict). Plus `reference.md`, the English lines, for the reference page.

**How to run**
1. `Runtime → Change runtime type → T4 GPU` (faster; CPU works too, just slower).
2. `Runtime → Run all`.
3. When asked, upload `data/raw/complaints_clean.csv` from your NanniSCII folder.
4. At the end, `nanniscii_translations.zip` downloads. Unzip it into `data/` in the project folder.

Same seed and same input file → same complaints, same lines.

In [ ]:
!pip -q install cuneiscribe==0.2.0 sentencepiece protobuf

In [ ]:
%%writefile nanni.py
"""
nanni.py: the shared helpers for turning complaint lines into cuneiform.

Used by translate.ipynb (Colab). No model is loaded here, so it can be
imported and tested anywhere.

  REDACTION              finds the CFPB's anonymised "XXXX" redactions
  pick_line()            chooses one line from a complaint
  translate_with_redactions()
                         translates around each redaction and keeps it as a
                         broken sign "x" (the scholarly mark for an illegible sign)
  normalize_atf()        praeclarum's CDLI spelling -> CuneiScribe spelling
  to_cuneiform()         Akkadian transliteration -> Unicode cuneiform
  coverage(), runaway()  checks on each translation
"""
import re

# CFPB redactions: XXXX, XX/XX/XXXX, {$XXXX}, XXXX-XXXX ...
REDACTION = re.compile(r"\{?\$?X{2,}(?:[/\-.]X{2,})*\}?")
GAP = "x"            # ATF convention: an illegible / broken sign
GAP_SIGN = "[x]"     # how a gap shows up in the cuneiform string

# ------------------------------------------------------------ choosing a line
SENTENCE_END = re.compile(r"(?<=[.!?])\s+|\n+")
BOILERPLATE = re.compile(
    r"^(dear|to whom|hello|hi\b|i am writing|i'm writing|please (see|find|help)|thank you|"
    r"sincerely|regards|re:|subject:|attached|this (is|letter) (a|to))", re.IGNORECASE)


def sentences(text):
    return [s.strip() for s in SENTENCE_END.split(text) if s and s.strip()]


def eligible(s, min_words=6, max_words=35, max_redactions=1):
    """A line worth translating: a real sentence, not a greeting, at most one redaction
    (more than one chops the line into fragments with no context, see Step 2b)."""
    n = len(s.split())
    return (min_words <= n <= max_words
            and len(REDACTION.findall(s)) <= max_redactions
            and len(REDACTION.sub("", s).split()) >= min_words - 1
            and not BOILERPLATE.match(s))


def pick_line(narrative, rng):
    """One random eligible line from a complaint, or None if it has none."""
    options = [s for s in sentences(narrative) if eligible(s)]
    return options[rng.integers(len(options))] if options else None


# ------------------------------------------------------------ translation
def normalize_atf(a):
    """praeclarum (CDLI ATF) spelling -> CuneiScribe spelling."""
    a = re.sub(r"_([^_]+)_", lambda m: m.group(1).upper(), a)   # _kug-ud_ -> KUG-UD (logograms)
    a = a.replace("_", "")
    a = re.sub(r"[\[\]⸢⸣#?!*<>]", "", a)                      # ATF damage/editorial marks
    for k, v in [("sz", "š"), ("SZ", "Š"), ("s,", "ṣ"), ("S,", "Ṣ"), ("t,", "ṭ"), ("T,", "Ṭ")]:
        a = a.replace(k, v)
    return a


def translate_with_redactions(line, to_akkadian, mode="split", raw=False):
    """Keep every redaction as one broken sign.
    mode="split": translate the text between redactions, rejoin with a gap (used).
    mode="single": collapse each redaction to a lone 'x' and translate the whole
    line (tested in Step 2b; it makes the model loop, so it is not used)."""
    if mode == "single":
        out = to_akkadian(REDACTION.sub("x", line))
        return out if raw else normalize_atf(out)
    parts = REDACTION.split(line)
    out = []
    for i, part in enumerate(parts):
        if part.strip(" ,.;:"):
            t = to_akkadian(part.strip())
            out.append(t if raw else normalize_atf(t))
        if i < len(parts) - 1:
            out.append(GAP)
    return " ".join(out)


def to_cuneiform(akk, conv):
    words = [GAP_SIGN if w == GAP else conv.to_cuneiform(w) for w in akk.split()]
    return " ".join(words)


# ------------------------------------------------------------ checks
def coverage(cuneiform):
    """Share of syllables that became real signs (CuneiScribe writes misses as [syllable])."""
    toks = re.findall(r"\[[^\]]+\]|[\U00012000-\U0001254F]", cuneiform)
    toks = [t for t in toks if t != GAP_SIGN]            # redactions aren't failures
    if not toks:
        return 0.0, []
    bad = [t for t in toks if t.startswith("[")]
    return round(100 * (1 - len(bad) / len(toks)), 1), bad


def runaway(akk, source):
    """True if the model looped: very long output or mostly one repeated word."""
    w = akk.split()
    return len(akk) > 4 * max(len(source), 20) or (len(w) > 6 and max(w.count(x) for x in set(w)) / len(w) > 0.5)


def signs(cuneiform):
    """The cuneiform signs in a string, in order (gaps and unmapped syllables left out)."""
    return re.findall(r"[\U00012000-\U0001254F]", re.sub(r"\[[^\]]*\]", "", cuneiform))

In [ ]:
# SETTINGS
N_LINES = 40                    # how many complaints (one line each)
SEED = 1                        # same seed -> same complaints
MODEL = "praeclarum/cuneiform"
INPUT = "complaints_clean.csv"

In [ ]:
# UPLOAD the cleaned complaints (data/raw/complaints_clean.csv)
import os
if not os.path.exists(INPUT):
    from google.colab import files
    up = files.upload()
    os.rename(next(iter(up)), INPUT)
import pandas as pd
pool = pd.read_csv(INPUT, dtype=str)
print(f"{len(pool):,} complaints in the pool")
pool.head(3)

In [ ]:
# PICK + TRANSLATE
import re, numpy as np
from cuneiscribe import AkkadianTranslator, CuneiformConverter, validate
from nanni import *

tr, conv = AkkadianTranslator(MODEL), CuneiformConverter()
ak = lambda s: tr.to_akkadian(s, max_length=128)    # cap length so a loop can't run away

ALLOWED = r"[^\w\s\-₀₁₂₃₄₅₆₇₈₉.,;:šṣṭḫŠṢṬḪāēīūÂÊÎÛàèìùáéíú]"
def safe_validate(a, line):
    # CuneiScribe's validator crashes on unexpected characters; catch it and report them
    try:
        return validate(a, line).suggestion, ""
    except TypeError:
        return "validator_crashed", "".join(sorted(set(re.findall(ALLOWED, a))))

rng = np.random.default_rng(SEED)
order = rng.permutation(len(pool))
rows, skipped = [], []
for k in order:
    if len(rows) == N_LINES:
        break
    c = pool.iloc[k]
    line = pick_line(c["narrative"], rng)
    if line is None:
        continue
    raw = translate_with_redactions(line, ak, raw=True)   # model's own spelling (for back-translation)
    a = normalize_atf(raw)                                 # CuneiScribe spelling (for signs)
    if runaway(a, line):                                   # a loop is not a translation: draw another
        skipped.append(c["complaint_id"]); continue
    cu = to_cuneiform(a, conv)
    cov, bad = coverage(cu)
    verdict, odd = safe_validate(a, line)
    rows.append(dict(
        n=len(rows) + 1, complaint_id=c["complaint_id"], date=c["date"], product=c["product"],
        english=line, akkadian_raw=raw, akkadian=a,
        back_translation=tr.to_english(raw, max_length=128),
        cuneiform=cu, n_signs=len(signs(cu)), gaps=cu.count(GAP_SIGN),
        sign_coverage_pct=cov, unmapped=" ".join(bad), validator=verdict, odd_chars=odd,
        low_confidence=(verdict != "render") or cov < 90))
    r = rows[-1]
    print(f"{r['n']:>2}. #{r['complaint_id']}  {r['english']}\n    {r['cuneiform']}\n    ↺ {r['back_translation']}\n")

In [ ]:
# TESTS
df = pd.DataFrame(rows)
all_signs = [s for c in df["cuneiform"] for s in signs(c)]
print(f"lines translated       {len(df)} / {N_LINES}")
print(f"skipped (model looped) {len(skipped)}")
print(f"signs in total         {len(all_signs):,}")
print(f"distinct signs         {len(set(all_signs))}   <- the renderer's alphabet")
print(f"redaction breaks       {df['gaps'].sum()}")
print(f"low confidence (kept, marked)  {df['low_confidence'].sum()}")
print(f"median sign coverage   {df['sign_coverage_pct'].median()}%")
print(f"seed fingerprint       {hash(tuple(df['complaint_id'])) % 10**8}   <- rerun: should be identical")
assert len(df) == N_LINES, "Not enough lines: lower N_LINES or check the input file"
assert df["cuneiform"].str.len().gt(0).all()
pd.set_option("display.max_colwidth", 80)
df[["n", "complaint_id", "english", "back_translation", "n_signs", "gaps", "sign_coverage_pct", "validator"]]

In [ ]:
# SAVE + DOWNLOAD
df.to_csv("translations.csv", index=False)
with open("reference.md", "w") as f:
    f.write("# The complaints\n\n")
    f.write("One line from each of these complaints was translated into Akkadian. "
            "Their signs are the alphabet the image is drawn with.\n\n")
    f.write("| # | Complaint | Received | Product | The line | What the machine says it wrote |\n|---|---|---|---|---|---|\n")
    for r in df.itertuples():
        esc = lambda s: str(s).replace("|", "\\|").replace("\n", " ")
        f.write(f"| {r.n} | {r.complaint_id} | {r.date} | {esc(r.product)} | {esc(r.english)} | {esc(r.back_translation)} |\n")
!zip -q nanniscii_translations.zip translations.csv reference.md
from google.colab import files
files.download("nanniscii_translations.zip")

## Then, on your Mac

Unzip into `data/`, and re-render the image from the real signs (from the NanniSCII folder, `.venv` active):

```
python3 generator/nanniscii_render.py "generator/images/Standing_male_worshiper_high_contrast.jpg" --text-csv data/translations.csv
python3 generator/nanniscii_vector.py "generator/images/Standing_male_worshiper_high_contrast.jpg" --text-csv data/translations.csv
```